In [1]:
import os
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv('/home/sally/myWork/SwitchAttention/industry/sic_us.csv')
print(df.head())


  ticker      cik     sic                 sicDescription entityType  \
0    FDP  1047340   100.0  Agricultural Production-Crops  operating   
1     BV  1734713   700.0          Agricultural Services  operating   
2    CLF   764065  1000.0                   Metal Mining  operating   
3    FCX   831259  1000.0                   Metal Mining  operating   
4   SCCO  1001838  1000.0                   Metal Mining  operating   

  stateOfIncorporation  
0                   E9  
1                   DE  
2                   OH  
3                   DE  
4                   DE  


In [6]:
import re
import pandas as pd

# ===== 參數 =====
INPUT_CSV = "/home/sally/myWork/SwitchAttention/industry/sic_us.csv"              # 換成你的檔名
OUT_COUNTS_CSV = "sic_desc_counts.csv"
OUT_CHECK_CSV  = "sic_code_to_desc_check.csv"

# ===== 工具：把 sicDescription 做一致化（避免大小寫/空白/符號差異）=====
def normalize_sic_desc(s: str) -> str:
    if pd.isna(s) or str(s).strip() == "":
        return "UNKNOWN"

    s = str(s).strip()

    # 常見清理：大小寫、空白、連字號/符號統一
    s = s.upper()
    s = s.replace("&", " AND ")
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", " ", s)       # 移除標點符號（保留字母/數字/底線/空白）
    s = re.sub(r"\s+", " ", s).strip()   # 多重空白壓成一個

    # 一些常見縮寫一致化（可視需要擴充）
    s = s.replace(" N E C", " NEC")      # "N.E.C." → "NEC"
    s = s.replace(" N E S", " NES")
    s = s.replace(" NOT ELSEWHERE CLASSIFIED", " NEC")

    return s

def main():
    # 讀資料（全部欄轉 str 可避免 int 轉換出錯）
    df = pd.read_csv(INPUT_CSV, dtype=str)

    # 基本資訊
    n_total = len(df)
    n_missing = df["sicDescription"].isna().sum() if "sicDescription" in df.columns else n_total
    print(f"總筆數：{n_total}，缺少 sicDescription：{n_missing}")

    # 新增標準化欄位
    df["sic_desc_norm"] = df["sicDescription"].map(normalize_sic_desc)

    # 類別總數
    n_classes = df["sic_desc_norm"].nunique(dropna=True)
    print(f"sicDescription（規整後）總共有 {n_classes} 類")

    # 每類公司數統計
    cnt = df["sic_desc_norm"].value_counts(dropna=False).rename_axis("sic_desc_norm").reset_index(name="count")
    print("\n前 20 大類別：")
    print(cnt.head(20))

    # 輸出統計
    cnt.to_csv(OUT_COUNTS_CSV, index=False)
    print(f"\n已輸出各類別計數到：{OUT_COUNTS_CSV}")

    # 檢查：同一個 sic 代碼是否對應多種描述（資料品質檢核）
    if "sic" in df.columns:
        check = (
            df.groupby(["sic", "sic_desc_norm"], dropna=False)["ticker"]
              .nunique()
              .reset_index(name="n_tickers")
              .sort_values(["sic", "n_tickers"], ascending=[True, False])
        )
        check.to_csv(OUT_CHECK_CSV, index=False)
        print(f"已輸出 sic 代碼 → 描述檢查表到：{OUT_CHECK_CSV}")
        # 若同一 sic 代碼對應到多個描述，檔裡會很明顯（可用來修正 mapping）

if __name__ == "__main__":
    main()

總筆數：1070，缺少 sicDescription：2
sicDescription（規整後）總共有 290 類

前 20 大類別：
                                       sic_desc_norm  count
0                      REAL ESTATE INVESTMENT TRUSTS    109
1                 FIRE MARINE AND CASUALTY INSURANCE     29
2                             STATE COMMERCIAL BANKS     28
3                          NATIONAL COMMERCIAL BANKS     25
4                    CRUDE PETROLEUM AND NATURAL GAS     24
5                                  ELECTRIC SERVICES     21
6                                  INVESTMENT ADVICE     20
7                     SERVICES BUSINESS SERVICES NEC     15
8                      SERVICES PREPACKAGED SOFTWARE     14
9                MOTOR VEHICLE PARTS AND ACCESSORIES     13
10              ELECTRIC AND OTHER SERVICES COMBINED     12
11    SURGICAL AND MEDICAL INSTRUMENTS AND APPARATUS     12
12                                OPERATIVE BUILDERS     12
13  SECURITY BROKERS DEALERS AND FLOTATION COMPANIES     11
14                       PHARMA

### id

In [ ]:
import pandas as pd

# 1) 產業名稱 → 整數 ID（290 類）
desc = df["sicDescription"].fillna("UNKNOWN").astype(str).str.strip()
uniq = sorted(desc.unique().tolist())
desc2id = {d:i for i,d in enumerate(uniq)}           # e.g. "METAL MINING" -> 37
df["industry_id"] = desc.map(desc2id)

df.to_csv("industry_id_mapping.csv", index=False)


#### symbol list + id

In [2]:
symbol = pd.read_csv('/home/sally/dataset/data_preprocessing/symbol_list.csv')
print(symbol)

       idx symbol
0        0    DDD
1        1    MMM
2        2    AOS
3        3   ATEN
4        4    AIR
...    ...    ...
1065  1065   YUMC
1066  1066    YUM
1067  1067    ZBH
1068  1068    ZTS
1069  1069    ZWS

[1070 rows x 2 columns]


In [10]:
import pandas as pd

# 檔名自行調整
SYMBOL_CSV = "/home/sally/dataset/data_preprocessing/symbol_list.csv"      # 內含一欄 'symbol' 或 'ticker'
SIC_CSV    = "/home/sally/myWork/SwitchAttention/industry/industry_id_mapping.csv"           # 內含 'ticker','sicDescription'，可能已經有 'industry_id'
OUT_CSV    = "symbol_list_industry_id.csv"

sym = pd.read_csv(SYMBOL_CSV, dtype=str)
sym["symbol"] = sym["symbol"].str.strip().str.upper()

# 讀 SIC 資料並準備 industry_id
sic = pd.read_csv(SIC_CSV, dtype=str)
sic["ticker"] = sic["ticker"].str.strip().str.upper()

if "industry_id" not in sic.columns:
    sic["sicDescription"] = sic["sicDescription"].fillna("UNKNOWN").astype(str).str.strip()
    sic["industry_id"], _ = pd.factorize(sic["sicDescription"], sort=True)

# 做成對照表：ticker -> industry_id
id_map = sic.drop_duplicates("ticker").set_index("ticker")["industry_id"]

# 直接 map 回去（順序不變，新增在最後一欄）
sym["industry_id"] = sym["symbol"].map(id_map)

# 若要把缺失補成 -1（可選）
# sym["industry_id"] = sym["industry_id"].fillna(-1)

sym.to_csv(OUT_CSV, index=False)
print("done ->", OUT_CSV)


done -> symbol_list_industry_id.csv


In [ ]:
import pandas as pd
from pathlib import Path

# 路徑自己改成你的資料夾
YEAR_DIR = "/home/sally/dataset/ticker/nyse/yearly_symbol_id"        # 放 2015_symbol_id.csv ... 2024_symbol_id.csv
SIC_CSV  = "/home/sally/myWork/SwitchAttention/industry/symbol_list_industry_id.csv"    

# 1) 讀 mapping，做 ticker -> industry_id 對照
mp = pd.read_csv(SIC_CSV, dtype=str)
cols = [c.lower() for c in mp.columns]
if "ticker" in cols:
    mp["ticker"] = mp[[c for c in mp.columns if c.lower()=="ticker"][0]]
elif "symbol" in cols:
    mp["ticker"] = mp[[c for c in mp.columns if c.lower()=="symbol"][0]]
else:
    raise ValueError("mapping 檔需要有 symbol 或 ticker 欄位")

mp["symbol"] = mp["symbol"].astype(str).str.strip().str.upper()
id_map = mp.drop_duplicates("symbol").set_index("symbol")["industry_id"]

# 2) 跑每一年的檔案，依 symbol(ticker) 補上 industry_id（加在最後一欄）
for y in range(2015, 2025):
    src = os.path.join(YEAR_DIR, f"{y}_symbol_id.csv")   # 若你的檔名是 2015_symbol_id.csv，自行改成 f"{y}_symbol_id.csv"
    if not src.exists():
        print(f"[skip] {src} 不在")
        continue

    df = pd.read_csv(src, dtype=str)

    # 找到 symbol/ticker 欄位
    cands = [c for c in df.columns if c.lower() in ("symbol", "ticker")]
    if not cands:
        raise ValueError(f"{src} 缺少 symbol/ticker 欄位")
    key = cands[0]
    df["symbol"] = df[key].astype(str).str.strip().str.upper()

    # map 並補 -1
    df["industry_id"] = df["symbol"].map(id_map).fillna("-1")

    # 存新檔（不改動原欄位順序，只在最後多一欄）
    out = src.with_name(src.stem + "_with_industry.csv")
    df.to_csv(out, index=False)
    miss = (df["industry_id"] == "-1").sum()
    print(f"[ok] {y}: {len(df)} rows，未對到 {miss} -> {out}")

TypeError: unsupported operand type(s) for /: 'str' and 'str'